In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/ameliamazzola/miniproject2
%cd miniproject2

import sys
sys.path.insert(0, '/content/miniproject2')

In [ ]:
!pip install ultralytics

In [ ]:
import os
from ultralytics import YOLO

MODEL_PATH  = "/content/miniproject2/model/model.pt"
VIDEO_FULL  = "/content/drive/MyDrive/Sequence 01.mp4"
VIDEO_SHORT = "/content/drive/MyDrive/sequence_01_short.mp4"
VIDEO_OUT   = "/content/drive/MyDrive/miniproject2/outputs/annotated_output.mp4"

os.makedirs("/content/drive/MyDrive/miniproject2/outputs", exist_ok=True)

print("Model exists?", os.path.exists(MODEL_PATH))
print("Video exists?", os.path.exists(VIDEO_FULL))

In [ ]:
import subprocess

result = subprocess.run([
    "ffmpeg", "-i", VIDEO_FULL,
    "-t", "60",
    "-c", "copy",
    VIDEO_SHORT
])

print("Short video saved to:", VIDEO_SHORT)
print("Exists?", os.path.exists(VIDEO_SHORT))

In [ ]:
model = YOLO(MODEL_PATH)
print("Model loaded successfully")

In [ ]:
import cv2
from collections import deque

VEHICLE_CLASSES = {0: "car", 1: "bus", 2: "van", 3: "others"}
COLORS = {0: (0,255,0), 1: (255,100,0), 2: (0,100,255), 3: (200,0,200)}
SCORE_THRESH = 0.25
FPS_WINDOW_SECONDS = 5

cap = cv2.VideoCapture(VIDEO_SHORT)
fps = cap.get(cv2.CAP_PROP_FPS) or 25
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
window_size = int(fps * FPS_WINDOW_SECONDS)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(VIDEO_OUT, fourcc, fps, (width, height))

frame_counts = deque()
max_load = 0
max_flow = 0
total_frames = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    total_frames += 1
    results = model(frame, verbose=False)[0]

    boxes  = results.boxes.xyxy.cpu().numpy()
    labels = results.boxes.cls.cpu().numpy().astype(int)
    scores = results.boxes.conf.cpu().numpy()

    # count vehicles above threshold
    count = int((scores >= SCORE_THRESH).sum())
    max_load = max(max_load, count)

    # sliding window traffic flow
    frame_counts.append(count)
    if len(frame_counts) > window_size:
        frame_counts.popleft()
    max_flow = max(max_flow, sum(frame_counts))

    # draw boxes
    for box, label, score in zip(boxes, labels, scores):
        if score < SCORE_THRESH:
            continue
        x1, y1, x2, y2 = map(int, box)
        cls_name = VEHICLE_CLASSES.get(label, "unknown")
        color = COLORS.get(label, (255,255,255))
        cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
        cv2.putText(frame, f"{cls_name} {score:.2f}",
                    (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX,
                    0.5, color, 1, cv2.LINE_AA)

    # overlay stats
    cv2.putText(frame, f"Vehicles: {count}",
                (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)
    cv2.putText(frame, f"Max load: {max_load}",
                (10,60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)
    cv2.putText(frame, f"Max flow ({FPS_WINDOW_SECONDS}s): {max_flow}",
                (10,90), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)

    out.write(frame)

cap.release()
out.release()
print("Done.")

In [ ]:
print("=" * 40)
print("TRAFFIC ANALYSIS RESULTS")
print("=" * 40)
print(f"Total frames processed:  {total_frames}")
print(f"Max road load:           {max_load} vehicles in one frame")
print(f"Max traffic flow:        {max_flow} vehicles in {FPS_WINDOW_SECONDS} seconds")

In [ ]:
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(VIDEO_OUT)
frames_to_show = [0, 50, 100, 200]

for fnum in frames_to_show:
    cap.set(cv2.CAP_PROP_POS_FRAMES, fnum)
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(12,6))
    plt.imshow(frame)
    plt.title(f"Frame {fnum}")
    plt.axis("off")
    plt.show()

cap.release()

In [ ]:
from google.colab import files
files.download(VIDEO_OUT)

In [ ]:
import cv2

# grab just the first frame
cap = cv2.VideoCapture(VIDEO_SHORT)
ret, frame = cap.read()
cap.release()

# run model on it with very low threshold
results = model(frame, verbose=True, conf=0.01)[0]

print("Number of detections:", len(results.boxes))
print("Scores:", results.boxes.conf.cpu().numpy())
print("Labels:", results.boxes.cls.cpu().numpy())